# 04 — M3: Transfer learning (ResNet18 / MobileNetV2 / EfficientNet-B0)

**Chủ: Phong Trần** · lý thuyết: `docs/LY_THUYET.md` phần 4 · code: `src/gtsrb/models/m3_transfer.py`

Bốn thứ phải giải thích được, notebook này chứng minh từng cái bằng số:

1. Vì sao tri thức ImageNet dùng được cho biển báo?
2. Vì sao phải upsample lên 224 (và **nó không thêm thông tin**)?
3. Discriminative learning rate — ba nhóm LR khác nhau
4. Hai pha freeze → unfreeze

Train trước: `make train-m3` (nên chạy trên Colab, ~45 phút cho cả 3 backbone)

In [ ]:
# Cho notebook thấy src/.  Chạy ô này đầu tiên.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)          # mọi đường dẫn trong repo là tương đối từ gốc

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from gtsrb import CLASS_NAMES, NUM_CLASSES
print("torch", torch.__version__, "| MPS", torch.backends.mps.is_available(),
      "| CUDA", torch.cuda.is_available())

In [ ]:
# Tiện ích dùng chung: tìm và nạp lại một run đã train
import glob, json
sys.path.insert(0, str(ROOT / "scripts"))
from evaluate import load_run, test_loader_for
from gtsrb.utils.seed import pick_device

device = pick_device("auto")

def find_runs(prefix=""):
    """Danh sách thư mục run có checkpoint, lọc theo tiền tố tên model."""
    dirs = [pathlib.Path(p) for p in sorted(glob.glob("artifacts/runs/*"))]
    return [d for d in dirs if (d / "best.pt").exists() and d.name.startswith(prefix)]

def latest(prefix):
    runs = find_runs(prefix)
    if not runs:
        raise FileNotFoundError(
            f"Chưa có run nào tên bắt đầu bằng {prefix!r}. Chạy: make train-{prefix[:2]}")
    return runs[-1]

print("Các run đã có:")
for d in find_runs(): print("  ", d.name)

## 1. Ba backbone — ba triết lý kiến trúc

In [ ]:
from gtsrb.models.registry import build_model, count_parameters
from gtsrb.deploy.speed import model_cost

rows = []
for bb in ("resnet18", "mobilenetv2", "effnetb0"):
    m = build_model(f"m3_{bb}", img_size=224, pretrained=False)   # False = nhanh, chỉ đếm
    cost = model_cost(m, 224)
    rows.append({"backbone": bb, **cost,
                 "số lớp conv": sum(1 for x in m.modules()
                                    if isinstance(x, torch.nn.Conv2d))})
cmp_m3 = pd.DataFrame(rows)
print(cmp_m3.to_string(index=False))
print()
print("Ý tưởng cốt lõi:")
print("  resnet18    : residual + basic block; conv DÀY ĐẶC, đơn giản, nhanh trên GPU")
print("  mobilenetv2 : depthwise separable + inverted residual + linear bottleneck")
print("  effnetb0    : compound scaling + MBConv + squeeze-and-excitation")

**★ Chú ý chỗ này — nó là bài học quan trọng nhất của phần triển khai.**

MobileNetV2 có FLOPs thấp hơn ResNet18 khoảng 6 lần. Nhưng latency thực tế **không**
nhanh hơn 6 lần, nhiều khi còn chậm hơn trên GPU. Ba nguyên nhân:

1. **Arithmetic intensity thấp** — FLOPs chỉ đếm phép tính, không đếm **truy cập bộ nhớ**.
   Depthwise conv làm rất ít phép tính trên mỗi byte đọc vào → bị chặn bởi **băng thông
   bộ nhớ**, không bởi năng lực tính toán. Giảm FLOPs không giúp gì khi nút cổ chai là bộ nhớ.
2. **Nhiều lần launch kernel** — nhiều lớp mảnh hơn, mỗi lớp có overhead cố định.
3. **Mức tối ưu thư viện** — cuDNN/BLAS tối ưu conv dày đặc rất kỹ; depthwise ít được chăm hơn.

`notebooks/09` sẽ **đo wall-clock** để xác nhận. Đừng suy latency từ FLOPs.

## 2. Vì sao phải upsample lên 224×224

Ba backbone này downsample tổng cộng **32 lần**.

In [ ]:
from gtsrb.explain.gradcam import feature_map_size
from gtsrb.utils.seed import pick_device

m = build_model("m3_resnet18", img_size=224, pretrained=False).to(device)
rows = []
for size in (48, 64, 112, 224):
    h, w = feature_map_size(m, size, device)
    rows.append({"input": f"{size}x{size}", "feature map cuối": f"{h}x{w}",
                 "số ô không gian": h * w,
                 "dùng được?": "KHÔNG — mất hết vị trí" if h <= 2
                               else ("tạm" if h < 7 else "ĐÚNG cấu hình pretrain")})
print(pd.DataFrame(rows).to_string(index=False))

**Nhưng phải trung thực về điều này:** nội suy 48→224 **KHÔNG THÊM THÔNG TIN**.

`notebooks/01` đã cho thấy trung vị kích thước ảnh gốc GTSRB chỉ khoảng **50×50 pixel** —
nên dù cache ở 224 cũng không có thêm bao nhiêu chi tiết thật.

Upsample để khớp **scale và stride** của mạng pretrained, **không** để có thêm chi tiết.
Nói được chỗ này là khác biệt giữa "học thuộc" và "hiểu".

Trục ablation 1 (64/112/224 cho M3) sẽ đo cái giá này:
`python scripts/run_ablation.py --axes resolution`

## 3. Discriminative learning rate

In [ ]:
m3 = build_model("m3_resnet18", img_size=224, pretrained=False)
groups = m3.param_groups(lr_early=1e-5, lr_middle=1e-4, lr_head=1e-3)

for g in groups:
    n = sum(p.numel() for p in g["params"])
    print(f"  nhóm {g['name']:7s}  lr={g['lr']:.0e}  {n:>10,} tham số")
print()
print("Vì sao tầng đầu LR nhỏ:")
print("  Tầng đầu học cạnh/màu RẤT TỐT từ 1,28 TRIỆU ảnh ImageNet.")
print("  Cho LR lớn = đập bỏ tri thức đó bằng tín hiệu từ 39 nghìn ảnh")
print("  -> CATASTROPHIC FORGETTING.")
print("  Tầng cuối mang ngữ nghĩa 'mèo/xe' nên phải đổi hoàn toàn -> LR lớn.")
print("  Cho toàn mạng 1e-3: phá pretrained. Cho toàn mạng 1e-5: head không hội tụ.")
print("  Hai cực đều tệ.")

## 4. Hai pha freeze → unfreeze

In [ ]:
import math
print(f"Loss ban đầu của head khởi tạo ngẫu nhiên ~ ln(43) = {math.log(43):.3f}")
print("-> gradient RẤT LỚN ở những bước đầu.")
print("Nếu backbone đang mở, gradient đó chảy ngược và PHÁ filter pretrained")
print("ngay trong vài trăm bước đầu, trước khi LR schedule kịp làm gì.\n")

m3.set_backbone_frozen(True)
frozen = count_parameters(m3, only_trainable=True)
m3.set_backbone_frozen(False)
unfrozen = count_parameters(m3, only_trainable=True)
print(f"Pha 1 (freeze)   : {frozen:>10,} tham số train được  ({frozen/unfrozen*100:.2f}%)")
print(f"Pha 2 (unfreeze) : {unfrozen:>10,} tham số train được  (100%)")
print("\nCấu hình 2 pha nằm trong configs/m3_resnet18.yaml -> train.phases")

In [ ]:
# Xem log train: LR và loss qua 2 pha
try:
    run = latest("m3_")
    h = pd.read_csv(run / "train_log.csv")
    print("run:", run.name)
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    for phase, grp in h.groupby("phase", sort=False):
        axes[0].plot(grp.epoch, grp.lr, marker="o", ms=3, label=phase)
        axes[1].plot(grp.epoch, grp.train_loss, marker="o", ms=3, label=phase)
        axes[2].plot(grp.epoch, grp.val_macro_f1, marker="o", ms=3, label=phase)
    axes[0].set_title("learning rate (nhóm head)"); axes[0].set_yscale("log")
    axes[1].set_title("train loss"); axes[2].set_title("val macro-F1")
    for ax in axes: ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print(h[["epoch","phase","lr","train_loss","val_top1","val_macro_f1"]].to_string(index=False))
except FileNotFoundError as e:
    print(e); print("\nChạy: make train-m3")

## 5. Chuẩn hoá — M3 dùng thống kê ImageNet, KHÔNG dùng của GTSRB

In [ ]:
from gtsrb import IMAGENET_MEAN, IMAGENET_STD
from gtsrb.utils.config import load_config

base = load_config("configs/base.yaml")
print("GTSRB  (dùng cho M1, M2 — train từ số 0, tính CHỈ trên tập train):")
print("  mean =", base.data.mean, "\n  std  =", base.data.std)
print("\nImageNet (dùng cho M3 — transfer learning):")
print("  mean =", list(IMAGENET_MEAN), "\n  std  =", list(IMAGENET_STD))
print("""
Hai lý do PHẢI dùng ImageNet stats cho M3:
  (a) filter tầng đầu được tối ưu cho đúng phân phối đầu vào đó;
  (b) running_mean/running_var trong các lớp BatchNorm pretrained được tích luỹ
      TRÊN phân phối đó -> đổi đầu vào là làm chúng sai.

Quên chuyện này là một LỖI IM LẶNG: model vẫn train bình thường, chỉ là kết quả tệ.
Vì vậy scripts/train.py TỰ khớp cấu hình dataset với model.normalize_mode
(xem src/gtsrb/models/registry.py).""")